In [ ]:
!uv add git+https://github.com/huggingface/diffusers
!uv add transformers accelerate bitsandbytes sentencepiece pillow ipywidgets

In [ ]:
import torch
from diffusers import QwenImageEditPipeline
from transformers import BitsAndBytesConfig as TransformersBitsAndBytesConfig
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig

print("Configuring quantization...")
# 1. Shrink the heavy text encoders and transformer
text_encoder_quantization_config = TransformersBitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

transformer_quantization_config = DiffusersBitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading Qwen-Image-Edit (This will take a moment)...")
# 2. Load the Pipeline (Do NOT use .to("cuda") here)
pipeline = QwenImageEditPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit",
    text_encoder_quantization_config=text_encoder_quantization_config,
    transformer_quantization_config=transformer_quantization_config,
    torch_dtype=torch.bfloat16,
)

print("Applying VRAM optimizations...")
# 3. Apply 10GB VRAM optimizations
# If you still get CUDA Out of Memory errors, change enable_model_cpu_offload() 
# to enable_sequential_cpu_offload() for maximum memory savings (at the cost of speed).
pipeline.enable_model_cpu_offload()
pipeline.vae.enable_tiling()
pipeline.vae.enable_slicing()

print("✅ Pipeline optimized and ready!")

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

# --- UI Elements ---
upload_widget = widgets.FileUpload(
    accept='image/*', 
    multiple=False,
    description='1. Upload Image',
    layout=widgets.Layout(width='auto')
)

prompt_widget = widgets.Textarea(
    value="Change the background to a futuristic cyberpunk city at night.",
    placeholder='Type your edit prompt here...',
    description='2. Prompt:',
    layout=widgets.Layout(width='80%', height='60px')
)

run_button = widgets.Button(
    description='3. Generate Edit',
    button_style='success',
    icon='check'
)

# Output area prevents the whole notebook cell from refreshing visually
output_area = widgets.Output()

# --- Execution Logic ---
def on_run_clicked(b):
    with output_area:
        clear_output(wait=True)
        
        if not upload_widget.value:
            print("⚠️ Please upload an image first!")
            return
            
        print(f"Processing prompt: '{prompt_widget.value}'...")
        
        # Extract image data from the widget (handles different ipywidgets versions)
        widget_val = upload_widget.value
        image_data = widget_val[0]['content'] if isinstance(widget_val, tuple) else list(widget_val.values())[0]['content']
        
        input_image = Image.open(io.BytesIO(image_data)).convert("RGB")
        
        inputs = {
            "image": input_image,
            "prompt": prompt_widget.value,
            "generator": torch.manual_seed(42), # You can randomize this if you want different variations
            "true_cfg_scale": 4.0,
            "negative_prompt": "",
            "num_inference_steps": 30, # 30 is a good balance of speed and quality
        }

        print("Generating edit... (Watch your terminal/console for the progress bar)")
        
        try:
            with torch.inference_mode():
                output = pipeline(**inputs)
                
            output_image = output.images[0]
            output_image.save("latest_edit.png")
            
            print("✅ Done! Saved as 'latest_edit.png'")
            display(output_image)
            
        except Exception as e:
            print(f"❌ An error occurred: {e}")

# --- Connect and Display ---
run_button.on_click(on_run_clicked)

print("🎨 Qwen-Image-Edit Studio")
display(upload_widget, prompt_widget, run_button, output_area)